In [32]:
import pickle
obj_bboxes_gt = pickle.load(open("/data/usrdata/shan/STAR_VideoQA/obj_bboxes_by_videoid.pkl",'rb'))
person_bboxes_gt = pickle.load(open("/data/usrdata/shan/STAR_VideoQA/person_bboxes_by_videoid.pkl",'rb'))
obj_bboxes_pred = pickle.load(open("/data/usrdata/shan/STTran_VG_charades_output/Full_Combined_Final_predicted_obj_bboxes_objthresh0.3_relthresh0.5.pkl",'rb'))
person_bboxes_pred = pickle.load(open('/data/usrdata/shan/STTran_VG_charades_output/Full_Combined_Final_predicted_person_bboxes.pkl','rb'))

In [34]:
def get_list_with_removed_elements(data_list,
                                  indices_to_remove_list
                                 ):
        return [element for index, element in enumerate(data_list) if index not in indices_to_remove_list]
    
def check_all_in_items_exist(in_items, or_list):
    return all(item in or_list for item in in_items)

def filter_mutually_exclusive_relations(or_obj_proposals_struct):
                                       #or_person_proposals_struct, 
                                       #min_rel_score_thresh, 
                                       #min_obj_score_thresh):
        num_att_rels_removed_total = 0
        num_spatial_rels_removed_total = 0
        num_contact_rels_removed_total = 0
        
        num_orig_att_rels = 0
        num_orig_spatial_rels = 0
        num_orig_contact_rels = 0
        
        for video_id, video_struct in or_obj_proposals_struct.items():
            #print('hello')
            for frame_id, obj_struct_list in video_struct['frames'].items():
                obj_indices_to_remove = []
                new_obj_struct_list = []
                #print(len(obj_struct_list))
                #input()
                for obj_i, obj_struct in enumerate(obj_struct_list):
                    if('attention_relationship_scores' in obj_struct):
                        rel_indices_to_remove = set()
                        mut_excl_items_to_check=[['looking_at', 'not_looking_at']]
                        num_orig_att_rels+= len(obj_struct['attention_relationship_scores'])
                        for mut_exl_set in mut_excl_items_to_check:
                            #print(obj_struct['attention_relationship'], mut_exl_set)
                            #input()
                            if(check_all_in_items_exist(mut_exl_set, obj_struct['attention_relationship'])):
                                #print('hello')
                                max_rel_score_index = 0
                                max_rel_score=0
                                covered_indices = []
                                for rel_name in mut_exl_set:
                                    rel_index = obj_struct['attention_relationship'].index(rel_name)
                                    covered_indices.append(rel_index)
                                    if(obj_struct['attention_relationship_scores'][rel_index] > max_rel_score):
                                        max_rel_score_index = rel_index
                                        max_rel_score = obj_struct['attention_relationship_scores'][rel_index]
                                less_score_indices = set(covered_indices) - set([max_rel_score_index])
                                #print('aa',less_score_indices)
                                rel_indices_to_remove = rel_indices_to_remove.union(less_score_indices)
                                #print('bb', rel_indices_to_remove)
                                #input()
                                #for rel_n in mut_exl_set:
                        obj_struct['attention_relationship_scores'] = get_list_with_removed_elements(obj_struct['attention_relationship_scores'], list(rel_indices_to_remove))
                        obj_struct['attention_relationship'] = get_list_with_removed_elements(obj_struct['attention_relationship'], list(rel_indices_to_remove))
                        #print(rel_indices_to_remove)
                        #if(len(rel_indices_to_remove)>0):
                         #   print('cc', obj_struct['attention_relationship'])
                        num_att_rels_removed_total+=len(rel_indices_to_remove)

                    if('spatial_relationship_scores' in obj_struct):
                        rel_indices_to_remove = set()
                        mut_excl_items_to_check=[['in_front_of', 'behind'],
                                                 ['above', 'beneath'],
                                                ]
                        num_orig_spatial_rels+= len(obj_struct['spatial_relationship_scores'])
                        
                        for mut_exl_set in mut_excl_items_to_check:
                            if(check_all_in_items_exist(mut_exl_set, obj_struct['spatial_relationship'])):
                                max_rel_score_index = 0
                                max_rel_score=0
                                covered_indices = []
                                for rel_name in mut_exl_set:
                                    rel_index = obj_struct['spatial_relationship'].index(rel_name)
                                    covered_indices.append(rel_index)
                                    if(obj_struct['spatial_relationship_scores'][rel_index] > max_rel_score):
                                        max_rel_score_index = rel_index
                                        max_rel_score = obj_struct['spatial_relationship_scores'][rel_index]
                                less_score_indices = set(covered_indices) - set([max_rel_score_index])
                                rel_indices_to_remove = rel_indices_to_remove.union(less_score_indices)
                                #for rel_n in mut_exl_set:
                        obj_struct['spatial_relationship_scores'] = get_list_with_removed_elements(obj_struct['spatial_relationship_scores'], list(rel_indices_to_remove))
                        obj_struct['spatial_relationship'] = get_list_with_removed_elements(obj_struct['spatial_relationship'], list(rel_indices_to_remove))
                        num_spatial_rels_removed_total+=len(rel_indices_to_remove)

                    if('contacting_relationship_scores' in obj_struct):
                        mut_excl_items_to_check=[['holding', 'not_contacting'],
                                                 ['not_contacting', 'touching'],
                                                 ['sitting_on', 'standing_on'],
                                                 ['sitting_on', 'carrying'],
                                                 ['standing_on', 'carrying']
                                                ]
                        
                        rel_indices_to_remove = set()
                        num_orig_contact_rels+= len(obj_struct['contacting_relationship_scores'])
                        
                        
                        for mut_exl_set in mut_excl_items_to_check:
                            if(check_all_in_items_exist(mut_exl_set, obj_struct['contacting_relationship'])):
                                max_rel_score_index = 0
                                max_rel_score=0
                                covered_indices = []
                                for rel_name in mut_exl_set:
                                    rel_index = obj_struct['contacting_relationship'].index(rel_name)
                                    covered_indices.append(rel_index)
                                    if(obj_struct['contacting_relationship_scores'][rel_index] > max_rel_score):
                                        max_rel_score_index = rel_index
                                        max_rel_score = obj_struct['contacting_relationship_scores'][rel_index]
                                less_score_indices = set(covered_indices) - set([max_rel_score_index])
                                rel_indices_to_remove = rel_indices_to_remove.union(less_score_indices)
                                #for rel_n in mut_exl_set:
                        obj_struct['contacting_relationship_scores'] = get_list_with_removed_elements(obj_struct['contacting_relationship_scores'], list(rel_indices_to_remove))
                        obj_struct['contacting_relationship'] = get_list_with_removed_elements(obj_struct['contacting_relationship'], list(rel_indices_to_remove))
                        num_contact_rels_removed_total+=len(rel_indices_to_remove)

                    new_obj_struct_list.append(obj_struct)
                    
                or_obj_proposals_struct[video_id]['frames'][frame_id] = new_obj_struct_list
                
        print(f"Filtered {num_att_rels_removed_total} number of att relations from original {num_orig_att_rels} ({100*num_att_rels_removed_total/num_orig_att_rels:.2f}%)")
        print(f"Filtered {num_spatial_rels_removed_total} number of spatial relations from original {num_orig_spatial_rels} ({100*num_spatial_rels_removed_total/num_orig_spatial_rels:.2f}%)")
        print(f"Filtered {num_contact_rels_removed_total} number of contact relations from original {num_orig_contact_rels} ({100*num_contact_rels_removed_total/num_orig_contact_rels:.2f}%)")
        return or_obj_proposals_struct

In [35]:
out1 = filter_mutually_exclusive_relations(obj_bboxes_gt, 
                                                     )

ZeroDivisionError: division by zero

In [33]:
obj_bboxes_pred['001YG.mp4']['frames']['000440.png']

[{'class': 'bag',
  'obj_score': array([0.92517334], dtype=float32),
  'bbox': (115.231, 130.8598, 104.285904, 119.809525),
  'attention_relationship': ['looking_at', 'not_looking_at'],
  'attention_relationship_scores': [2.9682302474975586, 0.8820754289627075],
  'spatial_relationship': ['in_front_of'],
  'spatial_relationship_scores': [1.0],
  'contacting_relationship': ['holding', 'touching'],
  'contacting_relationship_scores': [1.0, 1.0],
  'metadata': None,
  'visible': True},
 {'class': 'chair',
  'obj_score': array([0.99869245], dtype=float32),
  'bbox': (0.0, 219.6701, 197.44774, 50.82837),
  'attention_relationship': ['not_looking_at'],
  'attention_relationship_scores': [7.515574932098389],
  'spatial_relationship': ['beneath', 'behind'],
  'spatial_relationship_scores': [1.0, 1.0],
  'contacting_relationship': ['leaning_on',
   'other_relationship',
   'sitting_on'],
  'contacting_relationship_scores': [1.0, 0.5038344860076904, 1.0],
  'metadata': None,
  'visible': True},


In [30]:
out1['001YG.mp4']['frames']['000440.png']

[{'class': 'bag',
  'obj_score': array([0.92517334], dtype=float32),
  'bbox': (115.231, 130.8598, 104.285904, 119.809525),
  'attention_relationship': ['looking_at'],
  'attention_relationship_scores': [2.9682302474975586],
  'spatial_relationship': ['in_front_of'],
  'spatial_relationship_scores': [1.0],
  'contacting_relationship': ['holding', 'touching'],
  'contacting_relationship_scores': [1.0, 1.0],
  'metadata': None,
  'visible': True},
 {'class': 'chair',
  'obj_score': array([0.99869245], dtype=float32),
  'bbox': (0.0, 219.6701, 197.44774, 50.82837),
  'attention_relationship': ['not_looking_at'],
  'attention_relationship_scores': [7.515574932098389],
  'spatial_relationship': ['beneath', 'behind'],
  'spatial_relationship_scores': [1.0, 1.0],
  'contacting_relationship': ['leaning_on',
   'other_relationship',
   'sitting_on'],
  'contacting_relationship_scores': [1.0, 0.5038344860076904, 1.0],
  'metadata': None,
  'visible': True},
 {'class': 'laptop',
  'obj_score': ar